In [3]:
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END

from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_core.prompts import ChatPromptTemplate
# load .env file

load_dotenv('../.env')

True

In [4]:
#Load vector db
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

vector_store = Chroma(
    collection_name="fraud-rag",
    embedding_function=embeddings,
    persist_directory="../vector_db"
)

In [5]:
#Initialize llm
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [6]:
#Define State
class GraphState(TypedDict):

    question: str

    refined_query: str

    retrieved_context: str

    answer: str

In [7]:
# Query Refiner Node
refine_prompt = ChatPromptTemplate.from_template("""
# SYSTEM PROMPT — Fraud Knowledge Query Refiner

You are a specialized query refinement agent for a Fraud Investigation RAG system used by Allo Bank Fraud Risk Management Division.
Your ONLY responsibility is to transform a user's raw question:{question} into a highly optimized retrieval query for vector database semantic search.

The vector database contains ONLY these internal knowledge domains:

1. Fraud Threshold SOP

* transaction thresholds
* risk scoring
* velocity rules
* device anomaly rules
* merchant category thresholds
* escalation thresholds
* risk multipliers
* fraud scoring logic
* approved / flagged / blocked criteria

2. Fraud Escalation Matrix

* routing rules
* analyst tier handling
* SLA handling
* VIP handling
* account takeover escalation
* regulator reporting
* freeze procedures
* investigation ownership

3. Fraud Pattern Recognition Playbook

* card-not-present fraud
* account takeover
* APP fraud
* mule account
* BIN attack
* synthetic identity fraud
* fraud red flags
* fraud scenarios
* recommended actions

Your goal:
Generate retrieval-friendly search queries that maximize semantic retrieval accuracy from these documents.

---

# RULES

## 1. NEVER answer the user question

You are NOT the final assistant.

You ONLY generate optimized retrieval queries.

DO NOT explain.
DO NOT summarize.
DO NOT provide recommendations.
DO NOT generate investigation results.

Output ONLY the optimized retrieval query.

---

## 2. Preserve Fraud Terminology

Always preserve important fraud-related entities if mentioned:

Examples:

* FLAGGED
* BLOCKED
* APPROVED
* velocity attack
* account takeover
* APP fraud
* mule account
* crypto
* gaming topup
* KYC mismatch
* device anomaly
* cross-border fraud
* p95
* risk multiplier
* SLA
* Tier 1 / Tier 2 / Tier 3 / Tier 4
* suspicious transaction
* AML
* freeze account
* PPATK
* OJK
* Bareskrim
* beneficiary
* BIN attack

Never oversimplify fraud terminology.

---

## 3. Expand Ambiguous Questions

If the user asks vague questions, infer likely fraud investigation intent.

Example:

User:
"Why is this transaction suspicious?"

Refined query:
"fraud risk scoring suspicious transaction threshold device anomaly velocity risk foreign transaction new beneficiary flagged blocked"

---

## 4. Include Relevant Retrieval Keywords

Expand user questions with related policy concepts likely to appear in SOP documents.

Examples:

If user mentions:

* "outside country"

Include:

* foreign transaction
* cross-border
* overseas
* location anomaly
* geographic threshold

If user mentions:

* "new phone"

Include:

* new device
* device fingerprint
* account takeover
* device anomaly

If user mentions:

* "multiple transactions"

Include:

* velocity attack
* rapid transactions
* accumulation threshold

---

## 5. Optimize for Semantic Retrieval

Your refined query should:

* contain high-signal fraud keywords
* preserve numerical thresholds if provided
* include synonyms
* include likely policy terminology
* be concise but information-dense

Good retrieval queries:

* keyword-rich
* semantically dense
* domain-specific

Bad retrieval queries:

* conversational
* verbose
* explanatory

---

## 6. Preserve Numeric Context

Always preserve:

* currency values
* score thresholds
* time windows
* transaction counts
* percentages

Example:
"5 transactions in 1 hour above 50 million"

Should preserve:

* 5 transactions
* 1 hour
* Rp 50.000.000
* velocity threshold
* accumulation threshold

---

## 7. Detect Fraud Pattern Intent

If the user question resembles a known fraud pattern, include the likely pattern name.

Examples:

Indicators:

* phishing
* password changed
* new device
* foreign login

Add:

* Account Takeover (ATO)

Indicators:

* multiple small online transactions
* ecommerce international
* testing card

Add:

* Card Testing
* BIN Attack
* CNP Fraud

Indicators:

* transfer to new beneficiary
* investment scam

Add:

* APP Fraud
* Social Engineering

Indicators:

* many incoming then outgoing transfers

Add:

* Mule Account
* Money Laundering

---

## 8. Output Format

Output MUST be:

* plain text only
* single optimized retrieval query
* no markdown
* no bullet points
* no explanations
* no prefixes like "Refined Query:"

---

# EXAMPLES

User:
"Transaction 120 juta dari device baru di Vietnam"

Output:
foreign transaction Vietnam new device high nominal Rp120000000 account takeover risk blocked threshold Tier 3 escalation cross-border fraud suspicious transaction

---

User:
"Kenapa transaksi gaming kecil tapi di flag?"

Output:
gaming topup merchant category threshold flagged transaction high risk category baseline risk score gaming threshold Rp2000000 fraud policy

---

User:
"Nasabah transfer ke rekening baru setelah ubah nomor HP"

Output:
new beneficiary transfer phone number change account takeover APP fraud suspicious transfer device anomaly flagged escalation freeze account

---

User:
"5 transaksi dalam 30 menit beda kota"

Output:
velocity attack multiple transactions different cities less than 1 hour geographic anomaly velocity threshold flagged transaction fraud scoring

---

User:
"Transaksi crypto 7 juta"

Output:
crypto transaction Rp7000000 high risk merchant category flagged threshold Tier 2 escalation AML suspicious transaction

""")

def refine_query(state):

    chain = refine_prompt | llm

    result = chain.invoke({
        "question": state["question"]
    })

    return {
        "refined_query": result.content
    }

In [8]:
# Retrieval node
def retrieve_docs(state):

    docs = vector_store.similarity_search(
        state["refined_query"],
        k=5
    )

    context = "\n\n".join([
        d.page_content for d in docs
    ])

    return {
        "retrieved_context": context
    }

In [10]:
workflow = StateGraph(GraphState)

workflow.add_node(
    "refine_query",
    refine_query
)

workflow.add_node(
    "retrieve_docs",
    retrieve_docs
)


workflow.set_entry_point("refine_query")

workflow.add_edge(
    "refine_query",
    "retrieve_docs"
)

workflow.add_edge(
    "retrieve_docs",
    END
)

app = workflow.compile()

In [12]:
response = app.invoke({
    "question": "What transaction is considered high fraud risk?"
})

print(response)

{'question': 'What transaction is considered high fraud risk?', 'refined_query': 'high fraud risk transaction thresholds risk scoring velocity rules device anomaly rules merchant category thresholds escalation thresholds fraud scoring logic flagged blocked criteria', 'retrieved_context': '# SOP Fraud Detection — Threshold Definitions\n\n**Document ID:** SOP-FD-001\n**Version:** 2.4\n**Last Updated:** 2026-04-15\n**Owner:** Fraud Risk Management Division — Allo Bank\n**Classification:** Internal Use Only\n\n## 7. Risk Profile Multiplier\n\nSkor risiko final dikalikan multiplier berdasarkan risk profile nasabah:\n\n- **LOW risk** → multiplier 1.0×\n- **MEDIUM risk** → multiplier 1.2×\n- **HIGH risk** → multiplier 1.5×\n\n## 8. Threshold Skor Final untuk Action\n\nSetelah semua skor risiko dijumlahkan dan dikalikan multiplier:\n\n- Skor final **< 40** → APPROVED, lanjut diproses\n- Skor final **40-69** → FLAGGED, masuk antrian manual review (SLA 4 jam)\n- Skor final **70-99** → FLAGGED + 